# EEG Eye-State MLP Classifier + Optuna Hyperparameter Search

This notebook trains a **Multi-Layer Perceptron** to classify EEG recordings into *eyes open* vs *eyes closed*, using:

- **Band-power features** (Theta, Alpha, Beta, Gamma) + **Coefficient of Variation** per channel
- A **variable-depth perceptron** with configurable dropout and batch normalisation
- **Optuna** (TPE sampler + Median pruner) to find the best architecture and training hyperparameters automatically

> **Window rule:** a window is labelled *Open* if the eye was open at **any** sample within it.

---


## 1 · Imports & Setup

In [ ]:
# Use shared helpers from src package
from src import remove_artifacts_interpolate, band_power


In [1]:
import warnings

warnings.filterwarnings("ignore")

import logging
import os

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from scipy.io import arff
from scipy.signal import welch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib

matplotlib.use("Agg")  # remove if running interactively
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from mne.filter import filter_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}  |  Optuna: {optuna.__version__}")

Device : cpu
PyTorch: 2.11.0  |  Optuna: 4.8.0


## 2 · Load Dataset

The dataset is the **UCI EEG Eye State** dataset (Roesler, 2013).  
14 EEG channels sampled at 128 Hz, 14 980 observations.  
Target: `eyeDetection` — 0 = eyes closed, 1 = eyes open.

Place `EEG_Eye_State_Classification.csv` in the same directory as this notebook.


In [2]:
## IMPORT DATA
def load_arff(file_path):
    """Loads an ARFF file into a pandas DataFrame.

    Args:
        file_path (str): The path to the ARFF file.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the data from the ARFF file,
        or None if an error occurs.
    """
    if not file_path.lower().endswith(".arff"):
        logging.error(f"Invalid file type: {file_path}. Expected a .arff file.")
        return None

    if not os.path.exists(file_path):
        logging.error(f"File not found: {file_path}")
        return None

    try:
        logging.info("Loading data...")
        data, meta = arff.loadarff(file_path)
        df = pd.DataFrame(data)
        logging.info("Data loaded successfully.")
        return df
    except FileNotFoundError:
        logging.error(f"File not found: {file_path}")
    except Exception as e:
        logging.error(f"Error loading ARFF file: {e}")
    return None

In [4]:
print("Loading dataset …")

# data_path = "/Users/vinicius/.cache/kagglehub/datasets/robikscube/eye-state-classification-eeg-dataset/versions/1"
data_path = "../data/"

df = load_arff(os.path.join(data_path, "EEG_DATA.arff"))
df = pd.read_csv(f"{data_path}/session_test_2.csv", sep=";")

EEG_COLS = [c for c in df.columns if c != "eyeDetection" and c != "timestamp"]
TARGET = "eyeDetection"

print(f"  Shape : {df.shape}")
print(f"  Channels ({len(EEG_COLS)}): {EEG_COLS}")
print(f"  Class balance:\n{df[TARGET].value_counts()}\n")

ERROR:root:File not found: ../data/EEG_DATA.arff


Loading dataset …
  Shape : (21318, 18)
  Channels (16): ['ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7', 'ch8', 'ch9', 'ch10', 'ch11', 'ch12', 'ch13', 'ch14', 'ch15', 'ch16']
  Class balance:
eyeDetection
0    13003
1     8315
Name: count, dtype: int64



In [5]:
df = remove_artifacts_interpolate(df, EEG_COLS, 0.99, 0.01)

ch1     artifacts= 427  expanded= 498  [-63377.9, -48350.9]
ch2     artifacts= 426  expanded= 473  [-84187.0, -46804.6]
ch3     artifacts= 428  expanded= 489  [-39827.5, -10656.6]
ch4     artifacts= 427  expanded= 462  [-87401.7, -57903.5]
ch5     artifacts= 428  expanded= 465  [-63418.4, -34906.9]
ch6     artifacts= 426  expanded= 479  [-109188.4, -78289.7]
ch7     artifacts= 427  expanded= 469  [-80122.5, -49815.3]
ch8     artifacts= 427  expanded= 464  [-142236.2, -107007.3]
ch9     artifacts= 428  expanded= 513  [-77911.9, -49312.6]
ch10    artifacts= 428  expanded= 467  [-65615.6, -49695.0]
ch11    artifacts= 427  expanded= 466  [-102405.4, -78735.8]
ch12    artifacts= 428  expanded= 462  [-104442.2, -72647.7]
ch13    artifacts= 427  expanded= 460  [-114060.1, -76605.6]
ch14    artifacts= 427  expanded= 464  [-89764.7, -59119.1]
ch15    artifacts= 427  expanded= 449  [-77816.3, -48059.3]
ch16    artifacts= 428  expanded= 479  [-127863.1, -94048.3]


## 3 · Feature Extraction

For each **1-second window** (128 samples, 50 % overlap) we extract:

| Feature | Details |
|---|---|
| **Band power** | Welch PSD integrated over Theta (4–8 Hz), Alpha (8–13 Hz), Beta (13–30 Hz), Gamma (30–100 Hz) — one value per band per channel |
| **Coefficient of Variation** | σ/μ per channel — captures amplitude variability |

Total: 14 channels × (4 bands + 1 CV) = **70 features per window**.


## 4 · Preprocessing — StandardScaler + Torch Tensors

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np).astype(np.float32)

X_torch = torch.tensor(X_scaled)
y_torch = torch.tensor(y_np, dtype=torch.float32)

print(f"X_torch : {X_torch.shape}  dtype={X_torch.dtype}")
print(f"y_torch : {y_torch.shape}  dtype={y_torch.dtype}")

X_torch : torch.Size([330, 80])  dtype=torch.float32
y_torch : torch.Size([330])  dtype=torch.float32


## 5 · Variable-Depth MLP

The `MLP` class accepts a list of layer widths at construction time,  
making depth fully configurable. Each hidden layer optionally uses:

- **Batch Normalisation** (before activation)  
- **Dropout** (after activation) for regularisation


In [8]:
class MLP(nn.Module):
    """
    Flexible multi-layer perceptron.

    Parameters
    ----------
    in_dim      : number of input features
    layer_sizes : list of hidden-layer widths, e.g. [128, 64, 32]
    dropout_rate: dropout probability applied after each hidden activation
    activation  : one of 'relu', 'leaky_relu', 'elu', 'gelu'
    use_bn      : whether to insert BatchNorm1d before each activation
    """

    ACTIVATIONS = {
        "relu": nn.ReLU,
        "leaky_relu": nn.LeakyReLU,
        "elu": nn.ELU,
        "gelu": nn.GELU,
    }

    def __init__(
        self,
        in_dim: int,
        layer_sizes: list,
        dropout_rate: float,
        activation: str,
        use_bn: bool,
    ):
        super().__init__()
        Act = self.ACTIVATIONS[activation]
        layers = []
        prev = in_dim

        for size in layer_sizes:
            layers.append(nn.Linear(prev, size))
            if use_bn:
                layers.append(nn.BatchNorm1d(size))
            layers.append(Act())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            prev = size

        layers.append(nn.Linear(prev, 1))  # binary output (BCEWithLogits)
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)


# Quick sanity check
_demo = MLP(70, [128, 64], dropout_rate=0.3, activation="elu", use_bn=True)
print(_demo)
print(f"\nParameters: {sum(p.numel() for p in _demo.parameters()):,}")

MLP(
  (net): Sequential(
    (0): Linear(in_features=70, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ELU(alpha=1.0)
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

Parameters: 17,793


## 6 · Training & Evaluation Helpers

In [9]:
def train_epoch(model, loader, optimizer, criterion):
    """One full pass over the training loader."""
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()


@torch.no_grad()
def eval_auc(model, X_val: torch.Tensor, y_val: torch.Tensor) -> float:
    """Compute ROC-AUC on the validation set."""
    model.eval()
    loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
    probs = torch.cat(
        [torch.sigmoid(model(xb.to(DEVICE))).cpu() for xb, _ in loader]
    ).numpy()
    return float(roc_auc_score(y_val.numpy(), probs))


def build_optimizer(model, opt_name: str, lr: float, wd: float):
    if opt_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif opt_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        return torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=0.9)


def run_fold(
    X_tr,
    y_tr,
    X_va,
    y_va,
    layer_sizes,
    dropout,
    activation,
    use_bn,
    opt_name,
    lr,
    wd,
    batch_size,
    max_epochs=35,
    patience=7,
):
    """Train one fold; return (best_auc, predictions, probabilities, auc_history)."""
    pos_weight = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = MLP(X_scaled.shape[1], layer_sizes, dropout, activation, use_bn).to(DEVICE)
    optimizer = build_optimizer(model, opt_name, lr, wd)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    best_auc, best_state, pat_count, history = 0.0, None, 0, []

    for _ in range(max_epochs):
        train_epoch(model, loader, optimizer, criterion)
        auc = eval_auc(model, X_va, y_va)
        history.append(auc)

        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_count = 0
        else:
            pat_count += 1
        if pat_count >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=256)
        probs = torch.cat(
            [torch.sigmoid(model(xb.to(DEVICE))).cpu() for xb, _ in val_loader]
        ).numpy()

    preds = (probs >= 0.5).astype(int)
    return best_auc, preds, probs, history


print("Helpers defined.")

Helpers defined.


## 7 · Optuna Hyperparameter Search

### Search Space

| Hyperparameter | Range / Choices |
|---|---|
| `n_layers` | 1 – 5 |
| Hidden widths | 16 – 256 (log-scale); each layer ≤ previous (tapering) |
| `dropout` | 0.0 – 0.6 |
| `activation` | relu, leaky_relu, elu, gelu |
| `use_bn` | True / False |
| `optimizer` | adam, adamw, sgd |
| `lr` | 1e-4 – 1e-2 (log-scale) |
| `weight_decay` | 1e-5 – 1e-2 (log-scale) |
| `batch_size` | 32, 64, 128 |

**Sampler:** Tree-structured Parzen Estimator (TPE)  
**Pruner:** Median — prunes trials worse than the median at the same fold step  
**Objective:** mean ROC-AUC across 3-fold CV (fast inner loop)


In [10]:
N_TRIALS = 100
INNER_FOLDS = 3
cv_inner = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=42)


def objective(trial: optuna.Trial) -> float:
    # ── architecture ──────────────────────────────────────────────────
    n_layers = trial.suggest_int("n_layers", 1, 5)
    h0 = trial.suggest_int("h0", 32, 256, log=True)
    sizes = [h0]
    for i in range(1, n_layers):
        sizes.append(trial.suggest_int(f"h{i}", 16, sizes[-1]))

    dropout = trial.suggest_float("dropout", 0.0, 0.6)
    activation = trial.suggest_categorical(
        "activation", ["relu", "leaky_relu", "elu", "gelu"]
    )
    use_bn = trial.suggest_categorical("use_bn", [True, False])

    # ── optimiser ─────────────────────────────────────────────────────
    opt_name = trial.suggest_categorical("optimizer", ["adam", "adamw", "sgd"])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    wd = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # ── inner CV ──────────────────────────────────────────────────────
    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(cv_inner.split(X_scaled, y_np)):
        auc, _, _, _ = run_fold(
            X_torch[tr_idx],
            y_torch[tr_idx],
            X_torch[va_idx],
            y_torch[va_idx],
            sizes,
            dropout,
            activation,
            use_bn,
            opt_name,
            lr,
            wd,
            batch_size,
        )
        fold_aucs.append(auc)

        # Pruning: report intermediate value after each fold
        trial.report(np.mean(fold_aucs), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=0),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial : #{study.best_trial.number}")
print(f"Best AUC   : {study.best_value:.4f}")
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v}")

Best trial: 47. Best value: 0.950026: 100%|█| 10


Best trial : #47
Best AUC   : 0.9500

Best params:
  n_layers            : 5
  h0                  : 237
  h1                  : 205
  h2                  : 190
  h3                  : 175
  h4                  : 163
  dropout             : 0.04814396673870178
  activation          : leaky_relu
  use_bn              : True
  optimizer           : adam
  lr                  : 0.0007920420929019329
  weight_decay        : 0.0003734644549616174
  batch_size          : 32


## 8 · Extract Best Configuration

In [11]:
p = study.best_params
n_layers = p["n_layers"]
layer_sizes = [p["h0"]] + [p[f"h{i}"] for i in range(1, n_layers)]
dropout = p["dropout"]
activation = p["activation"]
use_bn = p["use_bn"]
opt_name = p["optimizer"]
lr = p["lr"]
wd = p["weight_decay"]
batch_size = p["batch_size"]

print("Best architecture")
print(
    f"  Input (70)  →  " + "  →  ".join(str(s) for s in layer_sizes) + "  →  Output (1)"
)
print(f"  Dropout     : {dropout:.3f}")
print(f"  Activation  : {activation}")
print(f"  Batch Norm  : {use_bn}")
print(f"  Optimizer   : {opt_name}")
print(f"  LR          : {lr:.2e}")
print(f"  Weight Decay: {wd:.2e}")
print(f"  Batch Size  : {batch_size}")

Best architecture
  Input (70)  →  237  →  205  →  190  →  175  →  163  →  Output (1)
  Dropout     : 0.048
  Activation  : leaky_relu
  Batch Norm  : True
  Optimizer   : adam
  LR          : 7.92e-04
  Weight Decay: 3.73e-04
  Batch Size  : 32


## 9 · Final Evaluation — 5-Fold Cross-Validation

Re-train the best configuration with **5-fold stratified CV**,  
collecting predictions and training curves for each fold.


In [12]:
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
final_aucs = []
all_preds = []
all_true = []
train_hists = []

for fold, (tr_idx, va_idx) in enumerate(cv_outer.split(X_scaled, y_np)):
    auc, preds, _, hist = run_fold(
        X_torch[tr_idx],
        y_torch[tr_idx],
        X_torch[va_idx],
        y_torch[va_idx],
        layer_sizes,
        dropout,
        activation,
        use_bn,
        opt_name,
        lr,
        wd,
        batch_size,
        max_epochs=50,
        patience=10,
    )
    final_aucs.append(auc)
    all_preds.extend(preds)
    all_true.extend(y_torch[va_idx].numpy().astype(int))
    train_hists.append(hist)
    print(f"Fold {fold+1}  AUC={auc:.4f}  epochs={len(hist)}")

print(f"\nMean AUC : {np.mean(final_aucs):.4f} ± {np.std(final_aucs):.4f}")
print(f"Accuracy : {(np.array(all_preds) == np.array(all_true)).mean():.4f}")
print()
print(classification_report(all_true, all_preds, target_names=["Closed", "Open"]))

Fold 1  AUC=0.9840  epochs=24
Fold 2  AUC=0.9793  epochs=26
Fold 3  AUC=0.9601  epochs=36
Fold 4  AUC=0.9810  epochs=23
Fold 5  AUC=0.9335  epochs=20

Mean AUC : 0.9676 ± 0.0190
Accuracy : 0.8970

              precision    recall  f1-score   support

      Closed       0.88      0.87      0.88       137
        Open       0.91      0.92      0.91       193

    accuracy                           0.90       330
   macro avg       0.89      0.89      0.89       330
weighted avg       0.90      0.90      0.90       330



## 10 · Plot A — Optuna Optimisation History

In [13]:
DARK = "#0a0e17"
CARD = "#111827"
EDGE = "#1f2937"
TEAL = "#00e5cc"
CORAL = "#ff4f5e"
GOLD = "#ffc947"
LIME = "#a8ff3e"
WHITE = "#f0f4ff"
GRAY = "#6b7280"

trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
comp = trials_df[trials_df["state"] == "COMPLETE"].sort_values("number")

vals = comp["value"].values
nums = comp["number"].values
best_sf = np.maximum.accumulate(vals)

fig, ax = plt.subplots(figsize=(12, 4), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

ax.scatter(nums, vals, color=TEAL, alpha=0.5, s=35, zorder=3, label="Trial AUC")
ax.plot(nums, best_sf, color=GOLD, lw=2.5, zorder=4, label="Best so far")
ax.axhline(
    best_sf[-1],
    color=CORAL,
    lw=1.2,
    ls="--",
    label=f"Best = {best_sf[-1]:.4f}",
    alpha=0.8,
)

ax.set_xlabel("Trial", color=WHITE, fontsize=11)
ax.set_ylabel("Validation AUC (3-fold)", color=WHITE, fontsize=11)
ax.set_title(
    "Optuna Optimisation History — TPE Sampler + Median Pruner",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE, lw=0.8)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=10)

plt.tight_layout()
plt.savefig("plot_optuna_history.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 11 · Plot B — Best Architecture Diagram

In [14]:
fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)
ax.axis("off")
ax.set_title(
    "Best MLP Architecture", color=WHITE, fontsize=14, fontweight="bold", pad=15
)

all_sz = [70] + layer_sizes + [1]
max_n = max(all_sz)
x_pos = np.linspace(0.05, 0.95, len(all_sz))
nc = [TEAL] + [LIME] * n_layers + [CORAL]
act_short = activation[:3].upper()
lbls = (
    ["Input\n(70)"]
    + [f"FC {s}\n{act_short}\ndrop={dropout:.2f}" for s in layer_sizes]
    + ["Output\n(1)\nσ"]
)

for i, (x, sz, col, lbl) in enumerate(zip(x_pos, all_sz, nc, lbls)):
    r = 0.03 + 0.07 * (sz / max_n)
    circle = mpatches.Circle(
        (x, 0.55), r, color=col, alpha=0.85, transform=ax.transAxes, zorder=3
    )
    ax.add_patch(circle)
    ax.text(
        x,
        0.55,
        str(sz),
        ha="center",
        va="center",
        color="black",
        fontsize=8,
        fontweight="bold",
        transform=ax.transAxes,
    )
    ax.text(
        x,
        0.13,
        lbl,
        ha="center",
        va="center",
        color=WHITE,
        fontsize=7.5,
        transform=ax.transAxes,
    )
    if i < len(all_sz) - 1:
        ax.annotate(
            "",
            xy=(x_pos[i + 1] - 0.03, 0.55),
            xytext=(x + r + 0.01, 0.55),
            xycoords="axes fraction",
            textcoords="axes fraction",
            arrowprops=dict(arrowstyle="->", color=GRAY, lw=1.8),
        )

meta = (
    f"Depth={n_layers}   Optimizer={opt_name.upper()}   "
    f"LR={lr:.1e}   BN={'Yes' if use_bn else 'No'}   "
    f"Dropout={dropout:.3f}"
)
ax.text(
    0.5,
    0.93,
    meta,
    ha="center",
    color=GOLD,
    fontsize=10,
    transform=ax.transAxes,
    fontweight="bold",
)

plt.tight_layout()
plt.savefig("plot_architecture.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 12 · Plot C — Per-Fold Training Curves

In [15]:
fold_colors = [TEAL, GOLD, LIME, CORAL, "#aa88ff"]

fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

for fi, hist in enumerate(train_hists):
    ax.plot(hist, color=fold_colors[fi], lw=2, alpha=0.9, label=f"Fold {fi+1}")

ax.set_xlabel("Epoch", color=WHITE, fontsize=11)
ax.set_ylabel("Validation AUC", color=WHITE, fontsize=11)
ax.set_title(
    "Training Curves — Best Config (5-fold CV)",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE, lw=0.8)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=10)

plt.tight_layout()
plt.savefig("plot_training_curves.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 13 · Plot D — Hyperparameter Analysis

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=DARK)
titles = [
    "Layer Depth Distribution",
    "Dropout Rate Distribution",
    "Learning Rate vs AUC",
]

for ax, title in zip(axes, titles):
    ax.set_facecolor(CARD)
    for s in ax.spines.values():
        s.set_color(EDGE)
    ax.tick_params(colors=WHITE)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight="bold")

# Panel 1: n_layers bar chart
col_nl = "params_n_layers"
if col_nl in comp.columns:
    nlv = comp[col_nl].dropna().astype(int)
    counts = nlv.value_counts().sort_index()
    bar_c = [GOLD if i == n_layers else TEAL for i in counts.index]
    axes[0].bar(counts.index, counts.values, color=bar_c, alpha=0.9, edgecolor=EDGE)
    axes[0].set_xlabel("# Hidden Layers", color=WHITE, fontsize=10)
    axes[0].set_ylabel("# Trials", color=WHITE, fontsize=10)
    axes[0].yaxis.grid(True, color=EDGE)
    axes[0].text(
        n_layers,
        counts[n_layers] + 0.3,
        "best",
        ha="center",
        color=GOLD,
        fontsize=9,
        fontweight="bold",
    )

# Panel 2: dropout histogram
col_dr = "params_dropout"
if col_dr in comp.columns:
    dv = comp[col_dr].dropna()
    axes[1].hist(dv, bins=16, color=LIME, alpha=0.75, edgecolor=EDGE)
    axes[1].axvline(
        dropout, color=CORAL, lw=2.5, ls="--", label=f"Best = {dropout:.3f}"
    )
    axes[1].set_xlabel("Dropout Rate", color=WHITE, fontsize=10)
    axes[1].set_ylabel("Count", color=WHITE, fontsize=10)
    axes[1].yaxis.grid(True, color=EDGE)
    axes[1].legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)

# Panel 3: LR scatter coloured by AUC
col_lr = "params_lr"
if col_lr in comp.columns:
    lrs = comp[col_lr].values
    auc_vals = comp["value"].values
    sc = axes[2].scatter(
        lrs, auc_vals, c=auc_vals, cmap="plasma", s=50, alpha=0.85, zorder=3
    )
    axes[2].axvline(lr, color=CORAL, lw=2.5, ls="--", label=f"Best = {lr:.1e}")
    axes[2].set_xscale("log")
    axes[2].set_xlabel("Learning Rate (log scale)", color=WHITE, fontsize=10)
    axes[2].set_ylabel("Validation AUC", color=WHITE, fontsize=10)
    axes[2].yaxis.grid(True, color=EDGE)
    axes[2].legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)
    plt.colorbar(sc, ax=axes[2], label="AUC")

plt.suptitle(
    "Hyperparameter Search Analysis",
    color=WHITE,
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig(
    "plot_hyperparam_analysis.png", dpi=150, bbox_inches="tight", facecolor=DARK
)
plt.show()

## 14 · Plot E — Confusion Matrix

In [17]:
fig, ax = plt.subplots(figsize=(5, 4), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

cm = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Closed", "Open"])
disp.plot(ax=ax, colorbar=False, cmap="YlOrRd")

ax.set_title(
    "Confusion Matrix — Aggregated CV Predictions",
    color=WHITE,
    fontsize=11,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.xaxis.label.set_color(WHITE)
ax.yaxis.label.set_color(WHITE)
for txt in ax.texts:
    txt.set_color("black")
    txt.set_fontsize(13)

plt.tight_layout()
plt.savefig("plot_confusion_matrix.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 15 · Results Summary

In [18]:
acc = (np.array(all_preds) == np.array(all_true)).mean()

print("=" * 50)
print("  EEG Eye-State MLP — Final Results")
print("=" * 50)
print(f"  Best trial AUC  : {study.best_value:.4f}")
print(f"  CV AUC (5-fold) : {np.mean(final_aucs):.4f} ± {np.std(final_aucs):.4f}")
print(f"  CV Accuracy     : {acc:.4f}")
print()
print("  Best Architecture")
print(
    f"    Input 70  →  " + "  →  ".join(str(s) for s in layer_sizes) + "  →  Output 1"
)
print(f"    Activation  : {activation}")
print(f"    Dropout     : {dropout:.3f}")
print(f"    Batch Norm  : {use_bn}")
print(f"    Optimizer   : {opt_name}  (lr={lr:.1e}, wd={wd:.1e})")
print(f"    Batch size  : {batch_size}")
print("=" * 50)

  EEG Eye-State MLP — Final Results
  Best trial AUC  : 0.9500
  CV AUC (5-fold) : 0.9676 ± 0.0190
  CV Accuracy     : 0.8970

  Best Architecture
    Input 70  →  237  →  205  →  190  →  175  →  163  →  Output 1
    Activation  : leaky_relu
    Dropout     : 0.048
    Batch Norm  : True
    Optimizer   : adam  (lr=7.9e-04, wd=3.7e-04)
    Batch size  : 32


#### Test on another dataset

In [19]:
data_path = "../data/"
df = pd.read_csv(f"{data_path}/session_test_3.csv", sep=";")

EEG_COLS = [c for c in df.columns if c != "eyeDetection" and c != "timestamp"]
TARGET = "eyeDetection"

df.iloc[:, 1:-1] = filter_data(df.iloc[:, 1:-1].values.T, FS, 1, 30).T

df = remove_artifacts_interpolate(df, EEG_COLS, 0.99, 0.01)

Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 423 samples (3.305 s)

ch1     artifacts= 432  expanded= 543  [-948.4, 872.9]
ch2     artifacts= 432  expanded= 637  [-559.3, 485.1]
ch3     artifacts= 432  expanded= 571  [-216.4, 200.7]
ch4     artifacts= 432  expanded= 590  [-235.4, 174.8]
ch5     artifacts= 432  expanded= 751  [-129.9, 120.6]
ch6     artifacts= 432  expanded= 683  [-233.1, 167.6]
ch7     artifacts= 432  expanded= 605  [-120.9, 144.3]
ch8     artifacts= 432  expanded= 645  [-112.0, 128.2]
ch9     artifacts= 432  expanded= 569  [-79

In [21]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np).astype(np.float32)

X_torch = torch.tensor(X_scaled)
y_torch = torch.tensor(y_np, dtype=torch.float32)

print(f"X_torch : {X_torch.shape}  dtype={X_torch.dtype}")
print(f"y_torch : {y_torch.shape}  dtype={y_torch.dtype}")

X_torch : torch.Size([169, 80])  dtype=torch.float32
y_torch : torch.Size([169])  dtype=torch.float32


In [22]:
model = MLP(X_torch.shape[1], layer_sizes, dropout, activation, use_bn).to(DEVICE)
optimizer = build_optimizer(model, opt_name, lr, wd)
loader = DataLoader(
    TensorDataset(X_torch, y_torch), batch_size=batch_size, shuffle=True
)

In [23]:
probs = torch.sigmoid(model(X_torch.to("cpu"))).cpu()

preds = (probs >= 0.5).to("cpu").int()

In [24]:
(preds == y_torch).sum() / len(y_torch)

tensor(0.4675)